# 01 · Carga y limpieza de la base de la SDM
**Grupo 2 · Estadística Bayesiana · Entrega final**  
Jean Paul Rojas Rojas · Daniel Estiven Pacheco Villagrande · Diego Andrés Cascante Camacho

**Qué hace.** Lee las cinco hojas de la base consolidada, traduce los códigos con el diccionario, revisa faltantes, repetidos y coherencia de la gravedad, construye las variables (franja, franja de 3 h, día, grave, con muertos…) y guarda `siniestros_limpios.csv`. Son las secciones 1 y 2 del notebook EDA original, sin cambios salvo la ruta de los archivos.

**Entradas.** `siniestros_viales_consolidados_bogota_dc.xlsx` (copia del equipo de la base de la SDM) en `notebooks/datos/`.

**Salidas.** `datos/siniestros_limpios.csv` (196.152 filas × 18 columnas).

*Respaldo por etapas: el notebook principal es `../Grupo2_Bayesiana_completo.ipynb`, que reúne estas ocho etapas.* Orden de ejecución y trazabilidad completa: `../README.md`.

In [1]:
# Configuración: funciona en Google Colab y en un computador
# En Colab: suba la carpeta "notebooks" a su Drive. Si el buscador automático no la encuentra,
# escriba su ruta en BASE_COLAB, p. ej. '/content/drive/MyDrive/Grupo2_EntregaFinal_Bayesiana/notebooks'
BASE_COLAB = ''
import os, sys, glob
from pathlib import Path
EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not BASE_COLAB:
        cand = glob.glob('/content/drive/MyDrive/**/notebooks/comun.py', recursive=True)
        assert cand, 'No encontré la carpeta notebooks en su Drive: escriba su ruta en BASE_COLAB'
        BASE_COLAB = os.path.dirname(cand[0])
    BASE = Path(BASE_COLAB)
else:
    # el notebook puede estar en notebooks/ o en notebooks/por_etapas/
    BASE = Path.cwd() if (Path.cwd() / 'comun.py').exists() else Path.cwd().parent
os.chdir(BASE)
sys.path.insert(0, str(BASE))
import numpy, pandas, scipy, matplotlib
print('Carpeta:', BASE)
print('Python', sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__,
      '| scipy', scipy.__version__, '| matplotlib', matplotlib.__version__)

Carpeta: C:\Users\dddan\Downloads\Pensamiento - Bayesiana\4. Estadística Bayesiana\Proyecto\4. Entrega final\notebooks
Python 3.12.10 | numpy 2.3.5 | pandas 2.3.3 | scipy 1.18.1 | matplotlib 3.10.7


In [2]:
if EN_COLAB:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'holidays'], check=True)
import holidays
from comun import ruta_datos, DATOS
CARPETA_TRABAJADOS = str(DATOS)   # aquí se guarda siniestros_limpios.csv
print('holidays', holidays.__version__)

holidays 0.108


In [3]:
# Encabezado del notebook EDA original (sección 0), sin la parte de rutas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
sns.set_theme(style='whitegrid', font_scale=0.95)
SEMILLA = 2026          # semilla del EDA original (Primera Entrega); el modelo final usa 20261109
rng = np.random.default_rng(SEMILLA)

NOMBRE = 'siniestros_viales_consolidados_bogota_dc.xlsx'
RUTA = ruta_datos(NOMBRE)          # notebooks/datos/ o la carpeta del proyecto
print('Leyendo', RUTA, '(tarda uno o dos minutos)')
hojas = pd.read_excel(RUTA, sheet_name=None)
sin, act, veh, hip, dic = (hojas[h].copy() for h in ['SINIESTROS', 'ACTOR_VIAL', 'VEHICULOS', 'HIPOTESIS', 'DICCIONARIO'])
display(pd.DataFrame({h: {'filas': d.shape[0], 'columnas': d.shape[1]} for h, d in hojas.items()}).T)

In [4]:
NOMBRE = 'siniestros_viales_consolidados_bogota_dc.xlsx'
RUTA = ruta_datos(NOMBRE)          # notebooks/datos/ o la carpeta del proyecto
print('Leyendo', RUTA, '(tarda uno o dos minutos)')
hojas = pd.read_excel(RUTA, sheet_name=None)
sin, act, veh, hip, dic = (hojas[h].copy() for h in ['SINIESTROS', 'ACTOR_VIAL', 'VEHICULOS', 'HIPOTESIS', 'DICCIONARIO'])
display(pd.DataFrame({h: {'filas': d.shape[0], 'columnas': d.shape[1]} for h, d in hojas.items()}).T)

Leyendo C:\Users\dddan\Downloads\Pensamiento - Bayesiana\4. Estadística Bayesiana\Proyecto\0. Base de Datos\siniestros_viales_consolidados_bogota_dc.xlsx (tarda uno o dos minutos)


,filas,columnas
SINIESTROS,196152,10
ACTOR_VIAL,422416,8
VEHICULOS,371605,7
HIPOTESIS,233819,3
DICCIONARIO,211,4


### 1.1 Traducción de códigos con el diccionario
Los códigos (localidad, gravedad, tipo de lugar, clase de vehículo) se traducen con la hoja DICCIONARIO y se comprueba que no quede ningún código sin traducir. Así ningún número queda "escrito a mano" en el código.

In [5]:
def nombres(campo, hoja='SINIESTROS'):
    d = dic[(dic.HOJA == hoja) & (dic.CAMPO == campo)]
    return dict(zip(d.CODIGO, d.DESCRIPCION))

LOCALIDAD = nombres('CODIGO_LOCALIDAD')
GRAVEDAD = nombres('GRAVEDAD')
CLASE_SIN = nombres('CLASE')
DISENO = nombres('DISENO_LUGAR')
CLASE_VEH = nombres('CLASE', 'VEHICULOS')

for campo, mapa in [('CODIGO_LOCALIDAD', LOCALIDAD), ('GRAVEDAD', GRAVEDAD), ('DISENO_LUGAR', DISENO), ('CLASE', CLASE_SIN)]:
    faltan = set(sin[campo].dropna().unique()) - set(mapa)
    assert not faltan, f'Códigos de {campo} sin traducir: {faltan}'
CODIGOS_MOTO = [c for c, n in CLASE_VEH.items() if str(n).lower().startswith('motocicl')]
print('Gravedad:', GRAVEDAD)
print('Códigos de moto según el diccionario:', {c: CLASE_VEH[c] for c in CODIGOS_MOTO})

Gravedad: {1: 'Con Muertos', 2: 'Con Heridos', 3: 'Solo Daños'}
Códigos de moto según el diccionario: {10: 'Motocicleta', 17: 'Motociclo'}


## 2. Limpieza y preparación
Cada decisión queda escrita con su razón.

### 2.1 ¿Hay datos faltantes?

In [6]:
faltan = pd.concat({h: d.isna().mean().mul(100) for h, d in
                    [('SINIESTROS', sin), ('ACTOR_VIAL', act), ('VEHICULOS', veh), ('HIPOTESIS', hip)]})
display(faltan[faltan > 0].round(1).rename('% faltante').to_frame())

% faltante
SINIESTROS CHOQUE           14.400
           OBJETO_FIJO      96.600
ACTOR_VIAL VEHICULO          5.600
VEHICULOS  CLASE             0.800
           SERVICIO          4.200
           MODALIDAD        61.300

**Qué vemos.** En los campos que usa el análisis (fecha, hora, localidad y gravedad) no falta nada. Los vacíos de `CHOQUE` y `OBJETO_FIJO` son estructurales: solo se llenan en ciertos tipos de siniestro. `MODALIDAD` está casi vacía, pero no se usa.

**Decisión.** No se rellena ni se borra nada.

### 2.2 ¿Hay siniestros repetidos?

In [7]:
print('Códigos de siniestro repetidos:', sin.CODIGO_ACCIDENTE.duplicated().sum())
print('Personas sin siniestro:', (~act.CODIGO_ACCIDENTE.isin(sin.CODIGO_ACCIDENTE)).sum())
print('Vehículos sin siniestro:', (~veh.CODIGO_ACCIDENTE.isin(sin.CODIGO_ACCIDENTE)).sum())

Códigos de siniestro repetidos: 0
Personas sin siniestro: 0
Vehículos sin siniestro: 0


**Decisión.** Cada siniestro aparece una vez, así que la **unidad de análisis es el siniestro**. Si se trabajara con personas o vehículos, un choque con varios involucrados contaría varias veces.

### 2.3 Variables de tiempo
- **Franja principal (4 bloques de 6 h):** madrugada (0–6), mañana (6–12), tarde (12–18), noche (18–24), fijada en el Avance 2.
- **Franja alternativa (8 bloques de 3 h):** prueba de sensibilidad.
- **Día en 4 grupos:** lunes a jueves, viernes, sábado, y domingo o festivo (con los festivos oficiales de Colombia).

In [8]:
sin['fecha'] = pd.to_datetime(sin.FECHA, format='%d/%m/%Y')
sin['anio'] = sin.fecha.dt.year
sin['hora'] = sin.HORA.str[:2].astype(int)
sin['minuto'] = sin.HORA.str[3:5].astype(int)
sin['hora_redondeada'] = sin.minuto.isin([0, 30])

FRANJAS4 = ['Madrugada (0-6)', 'Mañana (6-12)', 'Tarde (12-18)', 'Noche (18-24)']
sin['franja'] = pd.cut(sin.hora, [0, 6, 12, 18, 24], right=False, labels=FRANJAS4)
sin['franja3h'] = pd.cut(sin.hora, range(0, 25, 3), right=False, labels=[f'{h:02d}-{h+3:02d}' for h in range(0, 24, 3)])

festivos = set(holidays.Colombia(years=range(2015, 2021)).keys())
dia = sin.fecha.dt.dayofweek
es_festivo = sin.fecha.dt.date.isin(festivos)
sin['dia4'] = np.select([es_festivo | (dia == 6), dia == 5, dia == 4], ['Domingo o festivo', 'Sábado', 'Viernes'], 'Lunes a jueves')
assert sin.franja.notna().all()

print(f"Horas en punto o y media, 2015-2019: {sin.loc[sin.anio <= 2019, 'hora_redondeada'].mean():.1%} (si fueran exactas serían cerca de 3 %)")
print(f'Festivos que no caen en domingo, 2015-2020: {sum(1 for f in festivos if f.weekday() != 6)}')

Horas en punto o y media, 2015-2019: 38.9% (si fueran exactas serían cerca de 3 %)
Festivos que no caen en domingo, 2015-2020: 102


**Qué vemos.** Casi 4 de cada 10 horas están en punto o y media; si fueran exactas serían cerca del 3 % (2 de cada 60 minutos). Lo más probable es que muchas horas sean aproximadas; una explicación posible es que se anoten al diligenciar el reporte, pero los datos no permiten comprobarlo. Por eso no se usan cortes de menos de 3 horas.

**Nota de conciliación.** La bitácora reportó 37,1 % de graves para "domingo" (solo domingos). Aquí el grupo es "domingo o festivo" y da 37,3 %. No es un error: son dos definiciones distintas.

### 2.4 Variable respuesta
- `grave` = 1 si el siniestro dejó al menos un herido o un muerto; 0 si fue solo daños.
- `con_muertos` = 1 si dejó al menos un muerto (para la prueba de sensibilidad).

In [9]:
sin['grave'] = sin.GRAVEDAD.isin([1, 2]).astype(int)
sin['con_muertos'] = (sin.GRAVEDAD == 1).astype(int)
display(sin.GRAVEDAD.map(GRAVEDAD).value_counts().to_frame('siniestros'))

,siniestros
GRAVEDAD,
Solo Daños,127919
Con Heridos,65238
Con Muertos,2995


### 2.5 ¿La gravedad coincide con lo que les pasó a las personas?
Se separan dos situaciones distintas que antes estaban juntas:
- **Sin personas registradas.** Puede ser legítimo en un choque de solo daños (por ejemplo, contra un vehículo estacionado), pero no en uno marcado con heridos o muertos.
- **Gravedad que contradice el estado de las personas** (por ejemplo, marcado "con heridos" y todas las personas ilesas).

In [10]:
estado = act.groupby('CODIGO_ACCIDENTE').ESTADO.agg(
    lambda e: 1 if (e == 'MUERTO').any() else (2 if (e == 'HERIDO').any() else 3))
sin['gravedad_personas'] = sin.CODIGO_ACCIDENTE.map(estado)
sin['sin_personas'] = sin.gravedad_personas.isna()
sin['sin_personas_dudoso'] = sin.sin_personas & (sin.GRAVEDAD < 3)      # dice heridos/muertos pero no hay personas
sin['gravedad_contradice'] = ~sin.sin_personas & (sin.GRAVEDAD != sin.gravedad_personas)

display(pd.crosstab(sin.GRAVEDAD.map(GRAVEDAD), sin.sin_personas.map({True: 'Sin personas', False: 'Con personas'})))
print('Sin personas y marcados solo daños (posiblemente legítimos):', (sin.sin_personas & (sin.GRAVEDAD == 3)).sum())
print('Sin personas pero marcados con heridos o muertos (dudosos):', sin.sin_personas_dudoso.sum())
print('Con personas, pero la gravedad contradice su estado:', sin.gravedad_contradice.sum())

sin_personas,Con personas,Sin personas
GRAVEDAD,,
Con Heridos,65214,24
Con Muertos,2995,0
Solo Daños,127795,124


Sin personas y marcados solo daños (posiblemente legítimos): 124
Sin personas pero marcados con heridos o muertos (dudosos): 24
Con personas, pero la gravedad contradice su estado: 175


**Decisión.** Son menos de 2 de cada mil. Se dejan con su marca y se usa la gravedad oficial del siniestro, que es la que usa la Secretaría. Borrarlos sería decidir por nuestra cuenta cuál de los dos registros está mal.

### 2.6 Lugar
- **Localidad:** las 20, traducidas con el diccionario. Sumapaz no se borra; se marcará "no concluyente".
- **Tipo de lugar:** tramo de vía, intersección y otros (los otros 11 tipos tienen pocos casos). Solo para describir.

In [11]:
sin['localidad'] = sin.CODIGO_LOCALIDAD.map(LOCALIDAD)
nombre_lugar = sin.DISENO_LUGAR.map(DISENO)
sin['tipo_lugar'] = np.where(nombre_lugar.isin(['Tramo de Via', 'Interseccion']), nombre_lugar, 'Otros')
sin['tipo_lugar'] = sin.tipo_lugar.replace({'Tramo de Via': 'Tramo de vía', 'Interseccion': 'Intersección'})
assert sin.localidad.notna().all()
display(sin.tipo_lugar.value_counts().to_frame('siniestros'))

,siniestros
tipo_lugar,
Tramo de vía,150989
Intersección,38209
Otros,6954


### 2.7 Información que solo se conoce después del siniestro (fuga de información)
La clase de siniestro (choque, atropello…), con qué se chocó, si hubo moto o peatón, la causa probable y el estado de las personas describen lo que pasó **en** el siniestro. Se calculan para describir, pero **no entran al modelo**: la Secretaría no las conoce cuando planea, y algunas casi definen la gravedad (un atropello casi siempre deja un herido). Usarlas haría que el modelo "acierte" en el archivo y falle en la realidad.

In [12]:
con_moto = veh.loc[veh.CLASE.isin(CODIGOS_MOTO), 'CODIGO_ACCIDENTE'].unique()
con_peaton = act.loc[act.CONDICION == 'PEATON', 'CODIGO_ACCIDENTE'].unique()
sin['con_moto'] = sin.CODIGO_ACCIDENTE.isin(con_moto).astype(int)
sin['con_peaton'] = sin.CODIGO_ACCIDENTE.isin(con_peaton).astype(int)
sin['clase'] = sin.CLASE.map(CLASE_SIN)
print('Entran al modelo: localidad, franja y año')
print(f"% graves con moto: {sin.loc[sin.con_moto == 1, 'grave'].mean():.1%} | sin moto: {sin.loc[sin.con_moto == 0, 'grave'].mean():.1%}")

Entran al modelo: localidad, franja y año
% graves con moto: 72.9% | sin moto: 22.1%


### 2.8 Periodos
- **2015–2019 = periodo base.** 2020 se analiza aparte (la pandemia cambió la movilidad).
- **Ajuste 2015–2017 / prueba 2018–2019**, fijado en el Avance 2. Se parte por años y no al azar para imitar el uso real: con el pasado se decide sobre lo que viene.

In [13]:
sin['periodo'] = np.select([sin.anio <= 2017, sin.anio <= 2019], ['Ajuste 2015-2017', 'Prueba 2018-2019'], '2020 aparte')
base = sin[sin.anio <= 2019].copy()
ajuste_df = base[base.periodo == 'Ajuste 2015-2017']
cols = ['CODIGO_ACCIDENTE', 'fecha', 'anio', 'hora', 'franja', 'franja3h', 'dia4', 'localidad', 'tipo_lugar', 'grave',
        'con_muertos', 'sin_personas', 'sin_personas_dudoso', 'gravedad_contradice', 'con_moto', 'con_peaton', 'clase', 'periodo']
sin[cols].to_csv(os.path.join(CARPETA_TRABAJADOS, 'siniestros_limpios.csv'), index=False)
display(sin.periodo.value_counts().to_frame('siniestros'))

,siniestros
periodo,
Ajuste 2015-2017,101500
Prueba 2018-2019,71943
2020 aparte,22709


### Control de trazabilidad
Se compara el archivo recién generado con la copia que usó el equipo para el informe (si está disponible en la carpeta del proyecto).

In [14]:
import hashlib
nuevo = DATOS / 'siniestros_limpios.csv'
ref = PROYECTO_CSV = (Path(BASE).parent.parent / '0. Base de Datos' / 'datos_trabajados' / 'siniestros_limpios.csv')
print('Filas:', len(pd.read_csv(nuevo, usecols=['CODIGO_ACCIDENTE'])))
if ref.exists() and ref != nuevo:
    a, b = pd.read_csv(nuevo), pd.read_csv(ref)
    print('Idéntico a la copia del equipo:', a.equals(b))
else:
    print('No hay copia de referencia para comparar (normal en Colab).')

Filas: 196152


Idéntico a la copia del equipo: True
